# EgoScene v1: dataset tour

First-person walks rendered in UE5 with perfect ground truth: RGB, metric depth, semantic and instance segmentation, and camera poses.
This notebook loads a walk, shows one frame's ground truth side by side, back-projects it to a 3D point cloud, and fuses a whole walk into one cloud using the camera poses.

Run from the repo root's `dataset/` folder with the venv active (`pip install jupyter matplotlib`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from dataset.egoscene import list_walks, load_walk, load_frame, load_meta, backproject

DATA = os.path.abspath("../data")
walks = list_walks(DATA)
print(f"{len(walks)} walks, {sum(w['num_frames'] for w in walks)} frames")
for w in walks:
    print(f"  {w['walk_id']}  {w['level']:13s} {w['lighting']:9s} {w['speed_cm_s']:>4} cm/s  {w['num_frames']} frames")

## One frame: RGB, depth, semantic, instance

In [ ]:
WALK = os.path.join(DATA, "apartment_01", "walk_04")   # night walk
meta = load_meta(WALK)
fr = load_frame(WALK, 100, meta)
classes = meta["classes"]

cmap = plt.get_cmap("tab20", len(classes))
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0, 0].imshow(fr["rgb"]); ax[0, 0].set_title(f"RGB ({meta.get('lighting')})")
d = np.ma.masked_equal(fr["depth"], 0)
im = ax[0, 1].imshow(d, cmap="turbo"); ax[0, 1].set_title("Depth (m), invalid masked")
fig.colorbar(im, ax=ax[0, 1], fraction=0.03)
ax[1, 0].imshow(fr["semantic"], cmap=cmap, vmin=0, vmax=len(classes) - 1, interpolation="nearest")
ax[1, 0].set_title("Semantic")
ax[1, 1].imshow(fr["instance"], cmap="nipy_spectral", interpolation="nearest"); ax[1, 1].set_title("Instance")
for a in ax.ravel(): a.axis("off")
present = np.unique(fr["semantic"])
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(c)) for c in present]
ax[1, 0].legend(handles, [classes[c] for c in present], loc="center left", bbox_to_anchor=(-0.32, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

## Back-project the frame to 3D (world space, metres)

In [ ]:
K = np.array([[meta["intrinsics"]["fx"], 0, meta["intrinsics"]["cx"]],
              [0, meta["intrinsics"]["fy"], meta["intrinsics"]["cy"]], [0, 0, 1]])
pts, cols = backproject(fr["depth"], K, fr["c2w"], rgb=fr["rgb"])
sel = np.random.default_rng(0).choice(len(pts), 30000, replace=False)
fig = plt.figure(figsize=(9, 7)); ax = fig.add_subplot(projection="3d")
ax.scatter(*pts[sel].T, c=cols[sel] / 255.0, s=0.5)
cam = fr["c2w"][:3, 3]; ax.scatter(*cam, c="red", s=60, label="camera")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z (up)"); ax.legend()
ax.set_box_aspect(np.ptp(pts[sel], axis=0)); plt.show()
print(f"floor/ceiling height range: {pts[:, 2].min():.2f} .. {pts[:, 2].max():.2f} m")

## Fuse a whole walk with the camera poses

If depth, intrinsics and poses agree, frames from different viewpoints land on the same walls: a sharp top-down floor plan with the camera path through it.

In [ ]:
walk = load_walk(WALK, stride=20)
cloud = []
for i in range(len(walk["frame"])):
    p, _ = backproject(walk["depth"][i], walk["K"], walk["c2w"][i])
    cloud.append(p[np.random.default_rng(i).choice(len(p), 20000, replace=False)])
cloud = np.concatenate(cloud)
walls = (cloud[:, 2] > 0.3) & (cloud[:, 2] < 2.3)         # slice between floor and ceiling
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(cloud[walls, 0], cloud[walls, 1], s=0.2, c="k", alpha=0.3)
traj = walk["c2w"][:, :3, 3]
ax.plot(traj[:, 0], traj[:, 1], "r-", lw=2, label="camera path")
ax.set_aspect("equal"); ax.set_title(f"{os.path.basename(WALK)}: fused top-down view ({len(walk['frame'])} frames)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.legend(); plt.show()

## The four lighting presets

In [ ]:
by_light = {}
for w in walks:
    by_light.setdefault(w["lighting"], w)
fig, ax = plt.subplots(1, len(by_light), figsize=(5 * len(by_light), 3.2))
for a, (light, w) in zip(np.atleast_1d(ax), by_light.items()):
    f = load_frame(os.path.join(DATA, w["path"]), 100)
    a.imshow(f["rgb"]); a.set_title(f"{light} ({w['walk_id']})"); a.axis("off")
plt.tight_layout(); plt.show()